In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Load data
df = pd.read_csv('../data/processed/featured_data.csv')

# Separate features and target
feature_cols = [col for col in df.columns if col not in ['timestamp', 'target_temp_24h', 
                                                           'weather_description', 'country',
                                                           'latitude', 'longitude']]
X = df[feature_cols]
y = df['target_temp_24h']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split data (70% train, 30% test)
# IMPORTANT: For time-series, we should use the most recent data as test
# Sort by index (which corresponds to time) and take last 30%
split_idx = int(len(df) * 0.7)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"\nTrain set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler for later use
joblib.dump(scaler, '../data/models/scaler.pkl')
print("\n Data prepared and scaler saved")

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
import time

# Dictionary to store results
results = {}

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    """Train and evaluate a model"""
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    
    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Predictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Metrics
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    print(f"Training time: {training_time:.2f}s")
    print(f"Train MAE: {train_mae:.3f}°C | Test MAE: {test_mae:.3f}°C")
    print(f"Train RMSE: {train_rmse:.3f}°C | Test RMSE: {test_rmse:.3f}°C")
    print(f"Train R²: {train_r2:.3f} | Test R²: {test_r2:.3f}")
    
    results[name] = {
        'model': model,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'training_time': training_time,
        'predictions': y_pred_test
    }
    
    return model

# Train models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.1),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
}

for name, model in models.items():
    evaluate_model(name, model, X_train_scaled, X_test_scaled, y_train, y_test)

print("\n" + "="*50)
print("All models trained")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Test MAE': [results[m]['test_mae'] for m in results.keys()],
    'Test RMSE': [results[m]['test_rmse'] for m in results.keys()],
    'Test R²': [results[m]['test_r2'] for m in results.keys()],
    'Training Time (s)': [results[m]['training_time'] for m in results.keys()]
})

print("\n=== MODEL COMPARISON ===")
print(comparison_df.sort_values('Test MAE'))

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Test MAE
axes[0, 0].barh(comparison_df['Model'], comparison_df['Test MAE'])
axes[0, 0].set_xlabel('Mean Absolute Error (°C)')
axes[0, 0].set_title('Test MAE (Lower is Better)')
axes[0, 0].invert_yaxis()

# Test R²
axes[0, 1].barh(comparison_df['Model'], comparison_df['Test R²'])
axes[0, 1].set_xlabel('R² Score')
axes[0, 1].set_title('Test R² (Higher is Better)')
axes[0, 1].invert_yaxis()

# Test RMSE
axes[1, 0].barh(comparison_df['Model'], comparison_df['Test RMSE'])
axes[1, 0].set_xlabel('Root Mean Squared Error (°C)')
axes[1, 0].set_title('Test RMSE (Lower is Better)')
axes[1, 0].invert_yaxis()

# Training Time
axes[1, 1].barh(comparison_df['Model'], comparison_df['Training Time (s)'])
axes[1, 1].set_xlabel('Training Time (seconds)')
axes[1, 1].set_title('Training Time')
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=300)
plt.show()

# Find best model
best_model_name = comparison_df.sort_values('Test MAE').iloc[0]['Model']
best_model = results[best_model_name]['model']

print(f"\n🏆 Best Model: {best_model_name}")
print(f"Test MAE: {results[best_model_name]['test_mae']:.3f}°C")
print(f"Test R²: {results[best_model_name]['test_r2']:.3f}")

# Save best model
joblib.dump(best_model, '../data/models/best_model.pkl')
print(f"\n Best model saved to data/models/best_model.pkl")